# Privan-130M: Transformer Architecture Deep Dive

This notebook breaks down each component of Privan-130M:
- `TokenEmbedding` & `PositionEmbedding`
- `CausalSelfAttention` with PyTorch SDPA & fallback manual implementation
- `MLP` with GELU non-linearities
- `TransformerBlock` (Pre-LayerNorm residual flow)
- `CausalLM` with Weight Tying and output projection

In [ ]:
import sys
from pathlib import Path
import torch

sys.path.insert(0, str(Path.cwd().parent / "src"))

from llm.config import ModelConfig
from llm.model import TokenEmbedding, PositionEmbedding, CausalSelfAttention, MLP, TransformerBlock, CausalLM

## 1. Embeddings Test: [B, T] -> [B, T, D]

In [ ]:
B, T, D = 2, 8, 768
vocab_size = 50257

tok_emb = TokenEmbedding(vocab_size, D)
pos_emb = PositionEmbedding(1024, D)

sample_tokens = torch.randint(0, vocab_size, (B, T))
embeddings = tok_emb(sample_tokens) + pos_emb(T, device=sample_tokens.device)
print("Tokens shape:    ", sample_tokens.shape)
print("Embeddings shape:", embeddings.shape)

## 2. Multi-Head Self-Attention

In [ ]:
attn = CausalSelfAttention(embedding_dim=D, num_heads=12, context_length=1024)
attn_out, _ = attn(embeddings)
print("Attention Output shape:", attn_out.shape)

## 3. Full 130M Model Parameters

In [ ]:
cfg = ModelConfig()
model = CausalLM(cfg)
stats = model.count_parameters()
print(f"Total Trainable Parameters: {stats['total_trainable']:,} (~{stats['total_trainable']/1e6:.1f}M)")